In [78]:
from cfb.config import ROOT, SQL_ANALYSIS_DIR
from cfb.db import engine, read_query
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# modeling (scikit-learn)
import joblib                                                   # save / load the trained model
from sklearn.linear_model import LogisticRegression             # baseline model
from sklearn.ensemble import HistGradientBoostingClassifier     # main model: handles NULLs, monotonic constraints
from sklearn.pipeline import (make_pipeline, Pipeline)
from sklearn.impute import SimpleImputer                        # fill NULLs for the baseline (logistic can't take them)
from sklearn.preprocessing import (StandardScaler,SplineTransformer)
from sklearn.isotonic import IsotonicRegression                 # calibrate probabilities on the validation season
from sklearn.calibration import calibration_curve, CalibrationDisplay
from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer

from sklearn.metrics import (
    classification_report,
    ConfusionMatrixDisplay,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    roc_auc_score,
)

In [79]:
ep_query = '''
SELECT * FROM marts.expected_points
'''
df = pd.read_sql(ep_query, engine)
ep = df.copy()

In [80]:
ep.info()

<class 'pandas.DataFrame'>
RangeIndex: 2128731 entries, 0 to 2128730
Data columns (total 16 columns):
 #   Column             Dtype  
---  ------             -----  
 0   play_id            str    
 1   game_id            int64  
 2   season             int64  
 3   offense            str    
 4   defense            str    
 5   half               int64  
 6   sec_left_in_half   float64
 7   down               int64  
 8   distance           int64  
 9   yards_to_goal      int64  
 10  is_goal_to_go      bool   
 11  score_margin       int64  
 12  next_score_type    str    
 13  next_scoring_team  str    
 14  next_score_label   str    
 15  next_score_points  int64  
dtypes: bool(1), float64(1), int64(8), str(6)
memory usage: 245.6 MB


In [81]:
# Drop identifier columns
ep = ep.drop(columns=['play_id','game_id','offense','defense','next_scoring_team','next_score_type','next_score_points'],errors='ignore')

numerical_columns = ep.select_dtypes(include='number').columns.to_list()
bool_columns = ep.select_dtypes(include=['bool']).columns.to_list()
ep[bool_columns] = ep[bool_columns].astype('float')

target = ['next_score_label']

bool_features = [c for c in bool_columns if c not in target]

# Establish training dataset bounds
training = range(2014,2024)
validation = 2024
testing = 2025

train = ep[ep['season'].isin(training)]
valid = ep[ep['season'] == validation]
test = ep[ep['season'] == testing]

print(train.shape)
print(valid.shape)
print(test.shape)

(1167072, 9)
(120360, 9)
(123550, 9)


In [82]:
features = numerical_columns + bool_features
X_train, y_train = train[features], train[target]
X_valid, y_valid = valid[features], valid[target]
X_test, y_test = test[features], test[target]

In [83]:
numerical_pipeline = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler()
)

categorical_pipeline = make_pipeline(
    SimpleImputer(strategy='constant', fill_value=0)
)

preprocessor = ColumnTransformer([
    ('numeric', numerical_pipeline, numerical_columns),
    ('categoricals', categorical_pipeline, bool_columns)
])

logistic_regression_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ('model', LogisticRegression(max_iter=1000))   
])

In [86]:
logistic_regression_pipeline.fit(X_train, y_train)

# ---------- performance metrics (7-class target) ----------
y_true = y_valid.squeeze()                                         # one column -> Series
model_classes = logistic_regression_pipeline.classes_              # alphabetical; predict_proba columns follow this order

y_pred = logistic_regression_pipeline.predict(X_valid)
valid_proba = logistic_regression_pipeline.predict_proba(X_valid)  # all 7 columns, one per class

# per-class report (precision / recall are low for rare classes by nature; probabilities matter more for EP)
print(classification_report(y_true, y_pred, digits=4, zero_division=0))

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
sensitivity = recall_score(y_true, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

# multiclass log loss uses the full probability matrix
log_loss_score = log_loss(y_true, valid_proba, labels=model_classes)

# naive baseline: every snap gets the training class frequencies
train_freq = y_train.squeeze().value_counts(normalize=True).reindex(model_classes).fillna(0).to_numpy()
naive_log_loss = log_loss(y_true, np.tile(train_freq, (len(y_true), 1)), labels=model_classes)

# expected points: probability x points for each class, matched by class name
points_by_label = {'off_td': 7, 'off_fg': 3, 'off_safety': 2, 'no_score': 0,
                   'def_safety': -2, 'def_fg': -3, 'def_td': -7}
class_points = np.array([points_by_label[c] for c in model_classes])
ep_pred = valid_proba @ class_points
ep_actual = y_true.map(points_by_label).to_numpy()

print(f"Accuracy           : {accuracy:.4f}")
print(f"Precision (wtd)    : {precision:.4f}")
print(f"Recall (wtd)       : {sensitivity:.4f}")
print(f"F1 Score (wtd)     : {f1:.4f}")
print(f"Log Loss           : {log_loss_score:.4f}")
print(f"Naive Log Loss     : {naive_log_loss:.4f}")
print(f"Mean predicted EP  : {ep_pred.mean():.3f}  vs actual {ep_actual.mean():.3f}")
print()

# EP calibration: predicted EP vs actual next-score points, by predicted-EP band
ep_bands = pd.cut(ep_pred, bins=[-7, -1, 0, 1, 2, 3, 4, 5, 7])
ep_calibration = (
    pd.DataFrame({'band': ep_bands, 'predicted_ep': ep_pred, 'actual_points': ep_actual})
      .groupby('band', observed=True)
      .agg(snaps=('predicted_ep', 'size'), predicted_ep=('predicted_ep', 'mean'), actual_points=('actual_points', 'mean'))
)
ep_calibration['gap'] = ep_calibration['actual_points'] - ep_calibration['predicted_ep']
ep_calibration.round(3)

c:\Users\Owner\projects\cfb-project\.venv\Lib\site-packages\sklearn\utils\validation.py:1368: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


              precision    recall  f1-score   support

      def_fg     0.0000    0.0000    0.0000      7636
  def_safety     0.0000    0.0000    0.0000       317
      def_td     0.4191    0.2578    0.3192     22571
    no_score     0.5043    0.7289    0.5961     18247
      off_fg     0.4974    0.0046    0.0092     20531
  off_safety     0.0000    0.0000    0.0000       408
      off_td     0.5047    0.7964    0.6179     50650

    accuracy                         0.4948    120360
   macro avg     0.2751    0.2554    0.2203    120360
weighted avg     0.4523    0.4948    0.4118    120360

Accuracy           : 0.4948
Precision (wtd)    : 0.4523
Recall (wtd)       : 0.4948
F1 Score (wtd)     : 0.4118
Log Loss           : 1.2538
Naive Log Loss     : 1.4762
Mean predicted EP  : 1.890  vs actual 1.956



,snaps,predicted_ep,actual_points,gap
band,,,,
"(-7, -1]",6204,-1.787,-1.770,0.017
"(-1, 0]",11460,-0.421,-0.368,0.052
"(0, 1]",23864,0.524,0.567,0.043
"(1, 2]",23103,1.491,1.483,-0.008
"(2, 3]",20074,2.489,2.577,0.088
"(3, 4]",18051,3.491,3.559,0.068
"(4, 5]",12349,4.425,4.673,0.248
"(5, 7]",5255,5.483,5.548,0.065
